In [1]:
print("Hi")

Hi


In [2]:
from dotenv import load_dotenv
import os
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
# os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
# os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
print("Environment variables loaded successfully.")

Environment variables loaded successfully.


In [3]:
# =========================
# STEP 1: DEFINE TOOLS
# =========================

from langchain_openai import ChatOpenAI
from langchain.tools import tool

@tool
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b

@tool
def rag_search(query: str) -> str:
    """Answer questions about LangSmith"""
    return "LangSmith helps monitor, debug, and deploy LLM applications."

tools = [add, multiply, rag_search]


# =========================
# STEP 2: LLM WITH TOOLS
# =========================

llm = ChatOpenAI(model="gpt-4o-mini").bind_tools(tools, tool_choice="auto")

In [4]:
# =========================
# STEP 3: TOOL CALLING FLOW
# =========================

SYSTEM_PROMPT = """You are a helpful assistant with access to math tools.

IMPORTANT: For ANY math operations (addition, subtraction, multiplication, division, etc.), you MUST use the corresponding tool.
Do NOT calculate math in your head - always call the tool.

FORMAT RULE: Always respond in this format:
- For addition: "sum of X and Y = Z"
- For multiplication: "product of X and Y = Z"
- For other questions: Provide direct answer
Example: "sum of 5 and 7 = 12" """

def run_tool_calling(query: str):
    print("\n🧠 Question:", query)

    # Step 1: Ask LLM with system prompt
    response = llm.invoke([
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": query}
    ])

    # Step 2: Check if LLM wants to call a tool
    if response.tool_calls:
        print("🔧 Tool Call Detected:", response.tool_calls)

        tool_call = response.tool_calls[0]
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]
        tool_call_id = tool_call["id"]

        # Step 3: Execute tool
        for t in tools:
            if t.name == tool_name:
                tool_result = t.invoke(tool_args)
                break

        print("⚙️ Tool Result:", tool_result)

        # Step 4: Send result back to LLM
        final_response = llm.invoke([
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": query},
            response,
            {
                "role": "tool",
                "content": str(tool_result),
                "tool_call_id": tool_call_id
            }
        ])

        print("✅ Final Answer:", final_response.content)

    else:
        print("💬 Direct Answer:", response.content)

In [6]:
run_tool_calling("what is 5 + 7?")


🧠 Question: what is 5 + 7?
🔧 Tool Call Detected: [{'name': 'add', 'args': {'a': 5, 'b': 7}, 'id': 'call_Z0I50FKXVw4AMCprm2PGbe30', 'type': 'tool_call'}]
⚙️ Tool Result: 12
✅ Final Answer: sum of 5 and 7 = 12
